In [3]:
import os

In [4]:
%pwd

'd:\\MLops\\DataScienceProject_1\\Health_premium_calculator\\research'

In [5]:
os.chdir("../")

In [6]:
%pwd

'd:\\MLops\\DataScienceProject_1\\Health_premium_calculator'

In [7]:
import urllib.request as request
from src.datascience import logger
import zipfile

In [23]:
from dataclasses import dataclass
from pathlib import Path

@dataclass
class DataIngestionconfig:
    root_dir:Path
    local_data_file: Path
    unzip_dir: Path 

In [24]:
from src.datascience.constants import *
from src.datascience.utils.common import *

In [25]:
class ConfigurationManager:
    def __init__(self,
                 config_filepath = CONFIG_FILE_PATH,
                 params_filepath = PARAMS_FILE_PATH,
                 schema_filepath = SCHEMA_FILE_PATH) :
        
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        self.schema = read_yaml(schema_filepath)

        create_directories([self.config.artifacts_root]) 

    def get_data_ingestion_config(self) -> DataIngestionconfig :
        config = self.config.data_ingestion 
        create_directories([config.root_dir]) 


        data_ingestion_config = DataIngestionconfig(
            root_dir = config.root_dir,
            #source_URL = config.source_URL,
            local_data_file = config.local_data_file,
            unzip_dir = config.unzip_dir 
        )

        return data_ingestion_config 

In [28]:
import pandas as pd 

In [34]:
class DataIngestion:
    def __init__(self, config:DataIngestionconfig) :
        self.config = config 
    
    '''
    def download_file(self):
        if not os.path.exists(self.config.local_data_file):
            filename, headers = request.urlretrieve(
                url = self.config.source_URL,
                filename = self.config.local_data_file
            )
            logger.info(f"{filename} download! with following info: \n{headers}")
        else:
            logger.info(f"File already exists")
    '''

    def extract_zip_file(self):
        """
        zip_file_path: str
        Extracts the zip file into the data directory
        Function returns None
        """
        unzip_path = self.config.unzip_dir
        os.makedirs(unzip_path, exist_ok=True)
        with zipfile.ZipFile(self.config.local_data_file, 'r') as zip_ref:
            zip_ref.extractall(unzip_path)

    def convert_to_csv(self):
        unzip_path = self.config.unzip_dir  # This is a directory
        for file in os.listdir(unzip_path):
            if file.endswith(".xlsx"):
                xlsx_file_path = os.path.join(unzip_path, file)
                csv_path = os.path.join(unzip_path, "data.csv")

                df = pd.read_excel(xlsx_file_path)
                df.to_csv(csv_path, index=False)

                logger.info(f"Converted '{xlsx_file_path}' to '{csv_path}'")
                break
            else:
                logger.info("No .xlsx file found; no conversion needed.")
            


In [37]:
try:
    config=ConfigurationManager()
    data_ingestion_config=config.get_data_ingestion_config()
    data_ingestion=DataIngestion(config=data_ingestion_config)
    #data_ingestion.download_file()
    data_ingestion.extract_zip_file()
    data_ingestion.convert_to_csv() 
except Exception as e:
    raise e

[2025-04-30 12:21:15,101:INFO:common:yaml file: config\config.yaml loaded successfully]
[2025-04-30 12:21:15,104:INFO:common:yaml file: params.yaml loaded successfully]
[2025-04-30 12:21:15,107:INFO:common:yaml file: schema.yaml loaded successfully]
artifacts already exists, skipping.
artifacts/data_ingestion already exists, skipping.
[2025-04-30 12:21:27,822:INFO:3997881719:Converted 'artifacts/data_ingestion\premiums.xlsx' to 'artifacts/data_ingestion\data.csv']
